In [39]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchinfo import summary

In [ ]:
class UNET(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, features=(64, 128, 256, 512)):
        super().__init__()

        self.encoders = nn.ModuleList()
        current = in_channels
        for feature in features:
            self.encoders.append(self.conv_block(current, feature))
            current = feature

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = self.conv_block(features[-1], features[-1] * 2)

        self.up_convs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        
        for feature in reversed(features):
            self.up_convs.append(self.up_conv(feature * 2, feature))
            self.dec_blocks.append(self.conv_block(feature * 2, feature))

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def up_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        original = x
        skips = []

        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.up_convs, self.dec_blocks, reversed(skips)):
            x = up(x)            
            x = torch.cat((skip, x), dim=1)
            x = dec(x)

        mask = torch.sigmoid(self.final(x))
        return mask * original

In [43]:
model = UNET(in_channels=1, out_channels=2).cuda()
summary(model, input_size=(2,1 ,1040, 256))

Layer (type:depth-idx)                   Output Shape              Param #
UNET                                     [2, 2, 1040, 256]         --
├─ModuleList: 1-7                        --                        (recursive)
│    └─Sequential: 2-1                   [2, 64, 1040, 256]        --
│    │    └─Conv2d: 3-1                  [2, 64, 1040, 256]        576
│    │    └─BatchNorm2d: 3-2             [2, 64, 1040, 256]        128
│    │    └─LeakyReLU: 3-3               [2, 64, 1040, 256]        --
│    │    └─Conv2d: 3-4                  [2, 64, 1040, 256]        36,864
│    │    └─BatchNorm2d: 3-5             [2, 64, 1040, 256]        128
│    │    └─LeakyReLU: 3-6               [2, 64, 1040, 256]        --
├─MaxPool2d: 1-2                         [2, 64, 520, 128]         --
├─ModuleList: 1-7                        --                        (recursive)
│    └─Sequential: 2-2                   [2, 128, 520, 128]        --
│    │    └─Conv2d: 3-7                  [2, 128, 520, 128] 